# Transfer Learning
## Ejemplos 1 y 2 — un dataset real de Kaggle por ejercicio

| Ejercicio | Nivel | Dataset (Kaggle) | Modelo base | Enfoque |
|-----------|-------|------------------|-------------|---------|
| 1 | 🟢 Básico | Messy vs Clean Room (habitación desordenada vs. ordenada) | MobileNetV2 | Primer Feature Extraction |
| 2 | 🟡 Intermedio | Fruits Quality (fresca vs. podrida) | EfficientNetB0 | EDA + Data Augmentation + métricas  |

Cada ejercicio descarga y carga **su propio dataset** al comenzar.

## Configuración del Entorno

Verificamos GPU disponible e instalamos dependencias.

> **Kaggle:** los datasets se descargan con `kagglehub`. Si la descarga falla por credenciales, en Colab agrega los *Secrets* `KAGGLE_USERNAME` y `KAGGLE_KEY` (panel de la izquierda, activando "Notebook access") y vuelve a ejecutar.

In [1]:
# Verificar GPU
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True)
print("🖥️ GPU detectada:", gpu_info.stdout.strip() or "⚠️ No hay GPU — activa en Runtime > Change runtime type > T4 GPU")

🖥️ GPU detectada: Tesla T4, 15360 MiB


In [ ]:
!pip install -q kagglehub

import os, time, hashlib, collections
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import kagglehub
import warnings
warnings.filterwarnings('ignore')

# Configuración global
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU disponible: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 🧰 Funciones auxiliares (lectura de carpetas de imágenes)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp')

def carpetas_hoja(raiz):
    # Devuelve {carpeta: [rutas de imágenes]} solo para carpetas que contienen imágenes
    salida = {}
    for dp, _, files in os.walk(raiz):
        imgs = sorted(os.path.join(dp, f) for f in files if f.lower().endswith(IMG_EXT))
        if imgs:
            salida[dp] = imgs
    return salida

def mostrar_arbol(raiz, max_prof=4):
    # Árbol de carpetas con el número de imágenes directas de cada una
    base = raiz.rstrip(os.sep).count(os.sep)
    for dp, dirnames, files in os.walk(raiz):
        prof = dp.count(os.sep) - base
        if prof > max_prof:
            continue
        dirnames.sort()
        n_img = sum(f.lower().endswith(IMG_EXT) for f in files)
        extra = f"  [{n_img} imgs]" if files else ""
        print("  " * prof + "📁 " + (os.path.basename(dp) or dp) + extra)

def md5_archivo(ruta):
    h = hashlib.md5()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1 << 20), b''):
            h.update(bloque)
    return h.hexdigest()

def cargar_imagen(ruta, etiqueta):
    # Lee y decodifica una imagen desde disco (reemplaza a tfds.load del original)
    img = tf.io.read_file(ruta)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    return img, etiqueta

def imprimir_conclusion(titulo, filas, ancho=22, nota=None):
    # Caja de conclusión en texto: etiqueta ..... valor
    linea = '=' * 62
    print(linea)
    print(f" ✅ {titulo}")
    print(linea)
    for etiqueta, valor in filas:
        print((etiqueta + ' ').ljust(ancho, '.') + ' ' + str(valor))
    print(linea)
    if nota:
        print(nota)

def crear_dataset(rutas, etiquetas):
    ds = tf.data.Dataset.from_tensor_slices((list(rutas), np.array(etiquetas, dtype=np.int32)))
    return ds.map(cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE)

---
# 🟢 Ejemplo 1 — Básico: Habitación ordenada vs. desordenada con MobileNetV2

**Objetivo:** Primer contacto con Feature Extraction. Clasificación binaria (habitación ordenada `clean` vs. desordenada `messy`) usando MobileNetV2 pre-entrenado en ImageNet.

**Dataset:** [Messy vs Clean Room](https://www.kaggle.com/datasets/cdawn1/messy-vs-clean-room) (Kaggle) — según la inspección previa, 212 imágenes etiquetadas (carpetas `train` + `val`) y una carpeta `test` con 10 imágenes sin etiquetas.

**¿Por qué MobileNetV2?** Es ligero (~3.4M parámetros), rápido y perfecto para empezar.

### 1.1. Cargar y explorar el dataset

Primero **inspeccionamos la estructura real de carpetas** (sin suponer nada). El dataset no trae una división de prueba etiquetada y su `val` es muy pequeño, así que (igual que en el notebook original) uniremos las imágenes etiquetadas y haremos un split **80/20 con `SEED = 42`**.

In [ ]:
path_1 = kagglehub.dataset_download("cdawn1/messy-vs-clean-room")
print("Path to dataset files:", path_1)
print("\n🌳 Estructura real de carpetas:")
mostrar_arbol(path_1)

hojas_1 = carpetas_hoja(path_1)
print("\n📊 Imágenes por carpeta (hoja):")
for d, imgs in sorted(hojas_1.items()):
    print(f"  {len(imgs):>5}  {os.path.relpath(d, path_1)}")

In [ ]:
# Agrupar carpetas etiquetadas por (split, clase) = (carpeta padre, carpeta hoja)
SPLITS_ETIQUETADOS = ('train', 'val')   # 'test' no tiene subcarpetas de clase (sin etiquetas)
grupos_1 = collections.defaultdict(list)
for d in hojas_1:
    split, clase = os.path.basename(os.path.dirname(d)), os.path.basename(d)
    if split in SPLITS_ETIQUETADOS:
        grupos_1[(split, clase)].append(d)

CLASS_NAMES_1 = sorted({c for (_, c) in grupos_1})
NUM_CLASSES_1 = len(CLASS_NAMES_1)

# El árbol mostró carpetas repetidas (images/ e images/images/): eliminamos duplicados por contenido (MD5)
registro_1 = {}          # md5 -> (ruta, clase)
conteo_bruto = collections.Counter()
conflictos = 0
for (split, clase), carpetas in sorted(grupos_1.items()):
    for c in carpetas:
        for r in hojas_1[c]:
            conteo_bruto[(split, clase)] += 1
            h = md5_archivo(r)
            if h in registro_1:
                if registro_1[h][1] != clase:
                    conflictos += 1
            else:
                registro_1[h] = (r, clase)

print(f"📊 Clases: {CLASS_NAMES_1}")
print(f"📊 Imágenes etiquetadas (con posibles copias): {sum(conteo_bruto.values())}")
print(f"📊 Imágenes únicas (por contenido):            {len(registro_1)}")
print(f"📊 Duplicados descartados:                      {sum(conteo_bruto.values()) - len(registro_1)}")
print(f"⚠️ Mismo archivo con clases distintas:          {conflictos}")

# Carpeta test: ¿tiene etiquetas? (solo informativo, no se usa)
for d, imgs in hojas_1.items():
    if os.path.basename(d) == 'test':
        print(f"\n🔎 {os.path.relpath(d, path_1)} ({len(imgs)} imgs, sin subcarpetas de clase). Ejemplos de nombres:",
              [os.path.basename(p) for p in imgs[:5]])

In [ ]:
from sklearn.model_selection import train_test_split

rutas_1 = [v[0] for v in registro_1.values()]
etiq_1 = [CLASS_NAMES_1.index(v[1]) for v in registro_1.values()]

tr_p1, va_p1, tr_l1, va_l1 = train_test_split(
    rutas_1, etiq_1, test_size=0.20, random_state=SEED, stratify=etiq_1)

train_raw = crear_dataset(tr_p1, tr_l1)
val_raw = crear_dataset(va_p1, va_l1)

print(f"📊 Clases: {CLASS_NAMES_1}")
print(f"📊 Train: {tf.data.experimental.cardinality(train_raw).numpy():,} imágenes  "
      f"{dict(zip(CLASS_NAMES_1, np.bincount(tr_l1)))}")
print(f"📊 Val:   {tf.data.experimental.cardinality(val_raw).numpy():,} imágenes  "
      f"{dict(zip(CLASS_NAMES_1, np.bincount(va_l1)))}")

In [ ]:
# Visualizar muestras
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, (img, label) in enumerate(train_raw.take(10)):
    ax = axes[i // 5, i % 5]
    ax.imshow(img.numpy())
    ax.set_title(CLASS_NAMES_1[label.numpy()], fontsize=12,
                color='dodgerblue' if label.numpy() == 0 else 'tomato')
    ax.axis('off')
plt.suptitle('Messy vs Clean Room — Muestras del dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.2. Preprocesamiento

MobileNetV2 espera imágenes de **224×224** píxeles. Cada modelo tiene su propio `preprocess_input` que normaliza los píxeles al rango que usó durante su entrenamiento en ImageNet.

In [ ]:
IMG_SIZE_1 = (224, 224)
BATCH_SIZE = 32

# Función de preprocesamiento
def preprocess_example1(image, label):
    image = tf.image.resize(image, IMG_SIZE_1)
    # MobileNetV2 espera píxeles en [-1, 1]
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.one_hot(label, NUM_CLASSES_1)
    return image, label

# Crear pipelines
train_ds_1 = (train_raw
    .map(preprocess_example1, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

val_ds_1 = (val_raw
    .map(preprocess_example1, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

# Verificar un batch
for imgs, labels in train_ds_1.take(1):
    print(f"✅ Batch shape: {imgs.shape}")
    print(f"✅ Labels shape: {labels.shape}")
    print(f"✅ Rango de píxeles: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}]")

### 1.3. Construir modelo con Feature Extraction

**Pasos clave:**
1. Cargar MobileNetV2 **sin** el clasificador (`include_top=False`)
2. **Congelar** todas sus capas (`trainable = False`)
3. Agregar nuestro clasificador binario

In [ ]:
from tensorflow.keras import layers, models

# 1. Cargar backbone pre-entrenado (SIN clasificador de ImageNet)
base_model_1 = tf.keras.applications.MobileNetV2(
    weights='imagenet',       # Pesos entrenados en ImageNet
    include_top=False,        # Removemos las capas Dense de 1000 clases
    input_shape=(*IMG_SIZE_1, 3)
)

# 2. CONGELAR todo el backbone
base_model_1.trainable = False

print(f"📐 Backbone: MobileNetV2")
print(f"📐 Capas totales: {len(base_model_1.layers)}")
print(f"📐 Parámetros totales: {base_model_1.count_params():,}")
print(f"📐 Parámetros entrenables: 0  (todo congelado)")
print(f"📐 Output shape: {base_model_1.output_shape}")

In [ ]:
# 3. Construir modelo completo: backbone + nuevo clasificador
model_1 = models.Sequential([
    # Backbone congelado
    base_model_1,
    # Nuevo clasificador
    layers.GlobalAveragePooling2D(),     # Reduce 7×7×1280 → 1280
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES_1, activation='softmax')
], name='habitacion_mobilenetv2_FE')

model_1.summary()

# Contar parámetros
total = model_1.count_params()
trainable = sum(tf.size(w).numpy() for w in model_1.trainable_weights)
print(f"\n🔢 Total: {total:,} parámetros")
print(f"🔓 Entrenables: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"🔒 Congelados: {total - trainable:,} ({(total-trainable)/total*100:.2f}%)")

In [ ]:
print(f"💡 Observa: solo {trainable:,} parámetros son entrenables (el clasificador). "
      f"Los {total - trainable:,} del backbone están congelados. ¡Esto es Feature Extraction!")

### 1.4. Entrenar y evaluar

In [ ]:
# Compilar
model_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenar (pocas épocas porque Feature Extraction converge rápido)
t0 = time.time()
history_1 = model_1.fit(
    train_ds_1,
    validation_data=val_ds_1,
    epochs=8,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)
tiempo_1 = time.time() - t0

print(f"\n🏆 Mejor val_accuracy: {max(history_1.history['val_accuracy']):.4f}")
print(f"⏱️ Tiempo de entrenamiento: {tiempo_1:.0f} s")

In [ ]:
# Curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history_1.history['accuracy']) + 1)

axes[0].plot(epochs, history_1.history['accuracy'], 'o-', label='Train', color='#2196F3')
axes[0].plot(epochs, history_1.history['val_accuracy'], 's-', label='Val', color='#FF5722')
axes[0].set_title('Accuracy — Messy vs Clean Room (MobileNetV2 FE)', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.4, 1.02])   # rango ampliado: con pocos datos la accuracy puede ser menor al 80%

axes[1].plot(epochs, history_1.history['loss'], 'o-', label='Train', color='#2196F3')
axes[1].plot(epochs, history_1.history['val_loss'], 's-', label='Val', color='#FF5722')
axes[1].set_title('Loss — Messy vs Clean Room (MobileNetV2 FE)', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.5. Predicciones visuales

In [ ]:
# Predecir sobre muestras de validación
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, (img, label) in enumerate(val_raw.take(10)):
    ax = axes[i // 5, i % 5]
    # Preprocesar para predicción
    img_prep = tf.image.resize(img, IMG_SIZE_1)
    img_prep = tf.keras.applications.mobilenet_v2.preprocess_input(
        tf.expand_dims(img_prep, 0))
    pred = model_1.predict(img_prep, verbose=0)
    pred_class = CLASS_NAMES_1[np.argmax(pred)]
    true_class = CLASS_NAMES_1[label.numpy()]
    correct = pred_class == true_class

    ax.imshow(img.numpy())
    ax.set_title(f"Pred: {pred_class} ({pred.max():.0%})",
                fontsize=10, color='green' if correct else 'red',
                fontweight='bold')
    ax.set_xlabel(f"Real: {true_class}", fontsize=9)
    ax.axis('off')

plt.suptitle('Predicciones — Feature Extraction (MobileNetV2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Resultados reales del Ejemplo 1
val_loss_1, val_acc_1 = model_1.evaluate(val_ds_1, verbose=0)   # con los mejores pesos restaurados
mejor_acc_1 = max(history_1.history['val_accuracy'])
epocas_1 = len(history_1.history['accuracy'])
total_1 = model_1.count_params()
entren_1 = int(sum(tf.size(w).numpy() for w in model_1.trainable_weights))

imprimir_conclusion("CONCLUSIÓN EJERCICIO 1", [
    ("Dataset", f"Messy vs Clean Room ({NUM_CLASSES_1} clases: {', '.join(CLASS_NAMES_1)})"),
    ("Modelo base", f"MobileNetV2 ({base_model_1.count_params():,} params congelados)"),
    ("Params entrenados", f"{entren_1:,} ({entren_1/total_1*100:.2f}%)"),
    ("Imágenes train / val", f"{len(tr_p1)} / {len(va_p1)}"),
    ("Épocas ejecutadas", f"{epocas_1} de 8"),
    ("Mejor Val Accuracy", f"{mejor_acc_1:.4f}"),
    ("Val Accuracy final", f"{val_acc_1:.4f} (pesos restaurados)"),
    ("Test Accuracy", "N/D (la carpeta test no tiene etiquetas)"),
    ("Tiempo de entrenamiento", f"{tiempo_1:.0f} s"),
], nota=f"⚠️ Val pequeño ({len(va_p1)} imágenes): cada imagen vale {100/len(va_p1):.1f} puntos porcentuales.")

---
# 🟡 Ejemplo 2 — Intermedio: Frutas frescas vs. podridas con EfficientNetB0

**Objetivo:** Feature Extraction con clasificación binaria, EDA detallado, Data Augmentation, métricas avanzadas (confusion matrix, classification report) y visualización de feature maps.

**Dataset:** [Fruits Quality: Fresh vs Rotten](https://www.kaggle.com/datasets/nourabdoun/fruits-quality-fresh-vs-rotten) (Kaggle) — fotos de frutas en 2 clases: `fresh` (fresca) y `rotten` (podrida).

**¿Por qué EfficientNetB0?** Mejor balance accuracy/eficiencia de su generación. Un "upgrade" natural sobre MobileNetV2.

### 2.1. Cargar dataset y EDA

Inspeccionamos la estructura real y **verificamos el mapeo de clases**. En la inspección previa vimos que este dataset **ya viene con solo 2 clases** (`fresh` y `rotten`) y dividido en `train` / `valid` / `test`; por lo tanto **no hay que mover ni fusionar archivos**. La celda siguiente lo comprueba (solo lectura, por lo que es idempotente: se puede ejecutar varias veces sin riesgo).

In [ ]:
path_2 = kagglehub.dataset_download("nourabdoun/fruits-quality-fresh-vs-rotten")
print("Path to dataset files:", path_2)
print("\n🌳 Estructura real de carpetas:")
mostrar_arbol(path_2)

hojas_2 = carpetas_hoja(path_2)
df_orig = pd.DataFrame([{'split': os.path.basename(os.path.dirname(d)),
                         'clase_original': os.path.basename(d),
                         'imagenes': len(imgs)} for d, imgs in sorted(hojas_2.items())])
print("\n📊 Clases originales e imágenes por carpeta:")
display(df_orig)

In [ ]:
# --- Mapeo clase original -> clase final (basado en los nombres REALES de las carpetas) ---
clases_originales = sorted(df_orig['clase_original'].unique())
CLASES_FINALES = ['fresh', 'rotten']
MAPEO = {c: c for c in clases_originales}   # ya están en solo 2 clases: no hay nada que fusionar

print("🗺️ Tabla de mapeo (clase original → clase final):")
display(pd.DataFrame({'clase_original': list(MAPEO.keys()), 'clase_final': list(MAPEO.values())}))

# Si apareciera una clase inesperada, nos detenemos en vez de suponer
assert set(clases_originales) == set(CLASES_FINALES), (
    f"❌ Se esperaban {CLASES_FINALES} pero hay {clases_originales}. Revisar el mapeo antes de continuar.")

# --- Verificación ANTES vs DESPUÉS (sin mover archivos, por lo tanto nada se pierde ni se duplica) ---
antes = df_orig.groupby('clase_original')['imagenes'].sum()
despues = collections.Counter()
for d, imgs in hojas_2.items():            # segundo recorrido independiente aplicando el mapeo
    despues[MAPEO[os.path.basename(d)]] += len(imgs)

comparacion = pd.DataFrame({'antes': antes, 'despues': pd.Series(despues)})
comparacion.loc['TOTAL'] = comparacion.sum()
print("\n🔎 Conteo por clase final (antes vs. después):")
display(comparacion)
assert (comparacion['antes'] == comparacion['despues']).all(), "❌ Los conteos no coinciden"
print("✅ La suma coincide: no se perdió ningún archivo.")

# --- Duplicados por contenido (dentro de un split y entre splits) ---
hash_a_rutas = collections.defaultdict(list)
for d, imgs in hojas_2.items():
    for r in imgs:
        hash_a_rutas[md5_archivo(r)].append((os.path.basename(os.path.dirname(d)), os.path.basename(d)))
dups = {h: v for h, v in hash_a_rutas.items() if len(v) > 1}
cruzados = {h: v for h, v in dups.items() if len({s for s, _ in v}) > 1}
print(f"\n🔁 Archivos con contenido repetido: {len(dups)}  |  repetidos entre splits (posible fuga de datos): {len(cruzados)}")

# --- Balance entre clases ---
n_f, n_r = despues['fresh'], despues['rotten']
print(f"\n⚖️ Balance: fresh = {n_f}, rotten = {n_r}  (proporción {max(n_f, n_r)/min(n_f, n_r):.2f} : 1)")
print("✅ Clases prácticamente balanceadas" if max(n_f, n_r)/min(n_f, n_r) < 1.2
      else "⚠️ Clases DESBALANCEADAS — considerar class_weight o métricas por clase")

In [ ]:
# Cargar splits oficiales del dataset (train / valid / test)
def rutas_y_etiquetas(split):
    rutas, etiq = [], []
    for d, imgs in sorted(hojas_2.items()):
        if os.path.basename(os.path.dirname(d)) == split:
            rutas += imgs
            etiq += [CLASES_FINALES.index(MAPEO[os.path.basename(d)])] * len(imgs)
    return rutas, etiq

tr_p2, tr_l2 = rutas_y_etiquetas('train')
va_p2, va_l2 = rutas_y_etiquetas('valid')
te_p2, te_l2 = rutas_y_etiquetas('test')

train_raw_2 = crear_dataset(tr_p2, tr_l2)
val_raw_2 = crear_dataset(va_p2, va_l2)
test_raw_2 = crear_dataset(te_p2, te_l2)

NUM_CLASSES_2 = len(CLASES_FINALES)
CLASS_NAMES_2 = CLASES_FINALES
print(f"🍎 Clases ({NUM_CLASSES_2}): {CLASS_NAMES_2}")
print(f"📊 Train: {tf.data.experimental.cardinality(train_raw_2).numpy():,}")
print(f"📊 Val:   {tf.data.experimental.cardinality(val_raw_2).numpy():,}")
print(f"📊 Test:  {tf.data.experimental.cardinality(test_raw_2).numpy():,}")

In [ ]:
# Distribución de clases
train_labels = tr_l2
val_labels = va_l2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, labels, title_str in [(axes[0], train_labels, 'Train'),
                               (axes[1], val_labels, 'Validation')]:
    unique, counts = np.unique(labels, return_counts=True)
    colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES_2))
    bars = ax.bar([CLASS_NAMES_2[u] for u in unique], counts, color=colors, edgecolor='gray')
    ax.set_title(f'Distribución — {title_str}', fontweight='bold')
    ax.set_ylabel('Cantidad')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               str(count), ha='center', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Frutas fresh vs rotten — Distribución de clases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Muestras por clase
fig, axes = plt.subplots(NUM_CLASSES_2, 4, figsize=(14, 3 * NUM_CLASSES_2))

for class_idx in range(NUM_CLASSES_2):
    rutas_clase = [p for p, l in zip(tr_p2, tr_l2) if l == class_idx][:4]
    for col in range(4):
        ax = axes[class_idx, col]
        if col < len(rutas_clase):
            img, _ = cargar_imagen(rutas_clase[col], class_idx)
            ax.imshow(img.numpy())
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASS_NAMES_2[class_idx], fontsize=12, fontweight='bold',
                         rotation=0, labelpad=60, va='center')

plt.suptitle('Muestras por clase — Frutas fresh vs rotten', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 2.2. Preprocesamiento con Data Augmentation

Para mejorar la generalización, aplicamos **Data Augmentation solo al conjunto de entrenamiento**. EfficientNet tiene su propio `preprocess_input` que escala los píxeles de forma particular.

In [ ]:
IMG_SIZE_2 = (224, 224)

# Capa de augmentation (solo se aplica en entrenamiento)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augmentation')

def preprocess_train_2(image, label):
    image = tf.image.resize(image, IMG_SIZE_2)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image, training=True)
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    label = tf.one_hot(label, NUM_CLASSES_2)
    return image, label

def preprocess_val_2(image, label):
    image = tf.image.resize(image, IMG_SIZE_2)
    image = tf.keras.applications.efficientnet.preprocess_input(
        tf.cast(image, tf.float32))
    label = tf.one_hot(label, NUM_CLASSES_2)
    return image, label

train_ds_2 = (train_raw_2
    .map(preprocess_train_2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

val_ds_2 = (val_raw_2
    .map(preprocess_val_2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

test_ds_2 = (test_raw_2
    .map(preprocess_val_2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print("✅ Pipelines listos con augmentation en train")

In [ ]:
# Visualizar augmentation
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_img, sample_label = next(iter(train_raw_2))
sample_resized = tf.image.resize(tf.cast(sample_img, tf.float32), IMG_SIZE_2)

axes[0, 0].imshow(sample_img.numpy())
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

for i in range(1, 10):
    ax = axes[i // 5, i % 5]
    augmented = data_augmentation(tf.expand_dims(sample_resized, 0), training=True)
    ax.imshow(augmented[0].numpy().astype('uint8'))
    ax.set_title(f'Augmented #{i}', fontsize=10)
    ax.axis('off')

plt.suptitle(f'Data Augmentation — {CLASS_NAMES_2[sample_label.numpy()]}',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3. Construir modelo con EfficientNetB0

In [ ]:
# Backbone EfficientNetB0
base_model_2 = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE_2, 3)
)
base_model_2.trainable = False  # Congelar

# Modelo con clasificador más robusto
inputs_2 = layers.Input(shape=(*IMG_SIZE_2, 3))
x = base_model_2(inputs_2, training=False)  # BN en modo inferencia
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs_2 = layers.Dense(NUM_CLASSES_2, activation='softmax')(x)

model_2 = models.Model(inputs_2, outputs_2, name='frutas_efficientnetb0_FE')

# Resumen
total = model_2.count_params()
trainable = sum(tf.size(w).numpy() for w in model_2.trainable_weights)
print(f"📐 EfficientNetB0 — Feature Extraction")
print(f"🔢 Total: {total:,} | 🔓 Entrenables: {trainable:,} | 🔒 Congelados: {total-trainable:,}")

### 2.4. Entrenar con callbacks profesionales

In [ ]:
model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

t0 = time.time()
history_2 = model_2.fit(
    train_ds_2,
    validation_data=val_ds_2,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]
)
tiempo_2 = time.time() - t0

print(f"\n🏆 Mejor val_accuracy: {max(history_2.history['val_accuracy']):.4f}")
print(f"⏱️ Tiempo de entrenamiento: {tiempo_2:.0f} s")

In [ ]:
# Curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep = range(1, len(history_2.history['accuracy']) + 1)
axes[0].plot(ep, history_2.history['accuracy'], 'o-', label='Train', color='#4CAF50')
axes[0].plot(ep, history_2.history['val_accuracy'], 's-', label='Val', color='#E91E63')
axes[0].set_title('Accuracy — Frutas fresh vs rotten (EfficientNetB0 FE)', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history_2.history['loss'], 'o-', label='Train', color='#4CAF50')
axes[1].plot(ep, history_2.history['val_loss'], 's-', label='Val', color='#E91E63')
axes[1].set_title('Loss — Frutas fresh vs rotten (EfficientNetB0 FE)', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.5. Evaluación detallada: Confusion Matrix + Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Obtener predicciones
y_true, y_pred = [], []
for images, labels in val_ds_2:
    preds = model_2.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Classification Report
print("📊 Classification Report — Frutas fresh vs rotten (EfficientNetB0 FE) — Validación")
print("=" * 65)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES_2, digits=4))
reporte_val = classification_report(y_true, y_pred, target_names=CLASS_NAMES_2, output_dict=True)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=CLASS_NAMES_2, yticklabels=CLASS_NAMES_2, ax=ax,
           linewidths=0.5, linecolor='white',
           annot_kws={'size': 14, 'fontweight': 'bold'})

ax.set_xlabel('Predicción', fontsize=13, fontweight='bold')
ax.set_ylabel('Real', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix — Frutas fresh vs rotten (EfficientNetB0 FE)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top errores (umbral > 0 porque el conjunto de validación es pequeño)
print("\n🔍 Principales confusiones:")
hay_errores = False
for i in range(NUM_CLASSES_2):
    for j in range(NUM_CLASSES_2):
        if i != j and cm[i, j] > 0:
            hay_errores = True
            print(f"   {CLASS_NAMES_2[i]} → {CLASS_NAMES_2[j]}: {cm[i, j]} errores")
if not hay_errores:
    print("   (ninguna: no hubo errores en validación)")

In [ ]:
# Comprobación extra con el split `test` oficial del dataset (no usado en el entrenamiento)
yt_true, yt_pred = [], []
for images, labels in test_ds_2:
    preds = model_2.predict(images, verbose=0)
    yt_pred.extend(np.argmax(preds, axis=1))
    yt_true.extend(np.argmax(labels.numpy(), axis=1))
yt_true, yt_pred = np.array(yt_true), np.array(yt_pred)

print(f"📊 Classification Report — TEST ({len(yt_true)} imágenes)")
print("=" * 65)
print(classification_report(yt_true, yt_pred, target_names=CLASS_NAMES_2, digits=4))
print("Matriz de confusión (filas = real, columnas = predicción):")
print(confusion_matrix(yt_true, yt_pred))
test_acc_2 = float((yt_true == yt_pred).mean())

### 2.6. Visualización de Feature Maps

¿Qué "ve" el backbone EfficientNetB0 en nuestras frutas? Extraemos los mapas de activación de capas clave.

In [ ]:
# Seleccionar capas para visualizar
layer_names = ['block2a_expand_conv', 'block4a_expand_conv', 'block6a_expand_conv', 'top_conv']
feature_models = [
    models.Model(inputs=base_model_2.input,
                outputs=base_model_2.get_layer(name).output)
    for name in layer_names
]

# Tomar una imagen de ejemplo
sample_img_raw, sample_label_raw = next(iter(val_raw_2))
sample_processed = tf.image.resize(tf.cast(sample_img_raw, tf.float32), IMG_SIZE_2)
sample_processed = tf.keras.applications.efficientnet.preprocess_input(
    tf.expand_dims(sample_processed, 0))

# Extraer features
fig, axes = plt.subplots(len(layer_names), 9, figsize=(18, 2.5 * len(layer_names)))

for row, (feat_model, layer_name) in enumerate(zip(feature_models, layer_names)):
    features = feat_model.predict(sample_processed, verbose=0)
    # Mostrar la imagen original en la primera columna
    axes[row, 0].imshow(sample_img_raw.numpy())
    axes[row, 0].set_title('Original' if row == 0 else '', fontsize=9)
    axes[row, 0].set_ylabel(layer_name.replace('_', '\n'), fontsize=9,
                           fontweight='bold', rotation=0, labelpad=60, va='center')
    axes[row, 0].axis('off')
    # Mostrar 8 feature maps
    for col in range(1, 9):
        ax = axes[row, col]
        if col - 1 < features.shape[-1]:
            ax.imshow(features[0, :, :, col - 1], cmap='viridis')
        ax.axis('off')
        if row == 0:
            ax.set_title(f'Ch {col}', fontsize=9)

plt.suptitle(f'Feature Maps por Capa — {CLASS_NAMES_2[sample_label_raw.numpy()]}',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> 📝 **Observa la progresión**: las capas tempranas detectan bordes y texturas, las intermedias patrones más complejos (manchas, formas de la fruta), y las profundas abstracciones de alto nivel. Estos features "universales" son lo que transfiere ImageNet a nuestras frutas.

In [ ]:
# Resultados reales del Ejemplo 2
val_loss_2, val_acc_2 = model_2.evaluate(val_ds_2, verbose=0)   # con los mejores pesos restaurados
mejor_acc_2 = max(history_2.history['val_accuracy'])
epocas_2 = len(history_2.history['accuracy'])
total_2 = model_2.count_params()
entren_2 = int(sum(tf.size(w).numpy() for w in model_2.trainable_weights))
f1_macro = reporte_val['macro avg']['f1-score']

imprimir_conclusion("CONCLUSIÓN EJERCICIO 2", [
    ("Dataset", f"Fruits Quality ({NUM_CLASSES_2} clases: {', '.join(CLASS_NAMES_2)})"),
    ("Modelo base", f"EfficientNetB0 ({base_model_2.count_params():,} params congelados)"),
    ("Params entrenados", f"{entren_2:,} ({entren_2/total_2*100:.2f}%)"),
    ("Imágenes train/val/test", f"{len(tr_p2)} / {len(va_p2)} / {len(te_p2)}"),
    ("Épocas ejecutadas", f"{epocas_2} de 15"),
    ("Mejor Val Accuracy", f"{mejor_acc_2:.4f}"),
    ("Val Accuracy final", f"{val_acc_2:.4f} (pesos restaurados)"),
    ("F1-score macro (val)", f"{f1_macro:.4f}"),
    ("Test Accuracy", f"{test_acc_2:.4f}"),
    ("Tiempo de entrenamiento", f"{tiempo_2:.0f} s"),
], nota=f"⚠️ Val ({len(va_p2)}) y test ({len(te_p2)}) son pequeños: los resultados pueden variar entre ejecuciones.")

---
# 📋 Resumen final de los 2 ejercicios

In [ ]:
summary = pd.DataFrame([
    {'Ejercicio': '1 · Básico', 'Dataset': 'Messy vs Clean Room', 'Clases': NUM_CLASSES_1,
     'Modelo': 'MobileNetV2',
     'Val Acc': f"{max(history_1.history['val_accuracy']):.4f}",
     'Test Acc': 'N/D (test sin etiquetas)'},
    {'Ejercicio': '2 · Intermedio', 'Dataset': 'Fruits Quality (fresh vs rotten)', 'Clases': NUM_CLASSES_2,
     'Modelo': 'EfficientNetB0 + augmentation',
     'Val Acc': f"{max(history_2.history['val_accuracy']):.4f}",
     'Test Acc': f"{test_acc_2:.4f}"},
])
print(summary.to_string(index=False))

| Concepto | Descripción |
|----------|-------------|
| **Transfer Learning** | Reutilizar el conocimiento de ImageNet para nuestro problema |
| **Feature Extraction** | Congelar el backbone y entrenar solo el clasificador nuevo |
| `include_top=False` | Elimina las capas Dense originales de 1000 clases |
| `trainable = False` | Congela los pesos del backbone |
| `preprocess_input` | Cada modelo tiene su propia normalización |
| `training=False` | Las capas BatchNorm del backbone usan estadísticos de ImageNet |
| **Val vs Test** | Val sirve para ajustar/parar el entrenamiento; Test da la métrica final honesta |

**Siguiente paso natural:** *Fine-Tuning* — descongelar las últimas capas del backbone y re-entrenarlas con un learning rate muy bajo para adaptar los features a tu dominio.

---
*IS056B — Ingeniería del Conocimiento | UNCP 2026-II*